# Demostrador en Google Colab

Un modelo local de **Ollama** *propone* y un verificador externo *decide*:
- **Lean 4** para matemáticas (demostraciones formales).
- **SymPy** para derivaciones físicas.

Está pensado para problemas **largos**: guarda el progreso tras **cada intento** y, si se interrumpe (botón de parar, desconexión, límite de tiempo), al volver a ejecutar la celda **continúa donde se quedó**.

Menú *Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)*.

## 1. Instalar Ollama, Lean 4 y el modelo (una vez por sesión)

In [ ]:
# Dependencias y Ollama
!apt-get -qq install -y zstd pciutils lshw
!curl -fsSL https://ollama.com/install.sh | sh
!pip -q install sympy

import subprocess, time
subprocess.Popen(["ollama", "serve"])
time.sleep(6)

In [ ]:
# Lean 4 (vía elan)
!curl https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh -sSf | sh -s -- -y --default-toolchain stable
!/root/.elan/bin/lean --version

In [ ]:
MODELO = 'qwen2.5:7b'
!ollama pull {MODELO}

## 2. Descargar el proyecto y elegir dónde guardar el progreso

Con Google Drive montado, el progreso **sobrevive** aunque se cierre la sesión.

In [ ]:
!git clone -q https://github.com/Miguel283332/Orion.git 2>/dev/null || (cd Orion && git pull -q)
%cd Orion

from google.colab import drive
drive.mount('/content/drive')
DIR = '/content/drive/MyDrive/demostrador'      # estado y resultados
LEAN = '/root/.elan/bin/lean'

## 3. Problema fácil (matemáticas, Lean 4)

*«Demuestra la existencia de un número natural m mayor que cualquier n dado, tal que m sea par.»*

Si lo interrumpes con el botón de parar, el estado queda guardado: vuelve a ejecutar la celda para continuar.

In [ ]:
!python demostrador.py resolver --problema problemas/facil_par_mayor.json --modelo {MODELO} --lean {LEAN} --dir {DIR}/par_mayor

## 4. Problema fácil de física (SymPy): velocidad de escape y radio de Schwarzschild

In [ ]:
!python demostrador.py resolver --problema problemas/facil_velocidad_escape_schwarzschild.json --modelo {MODELO} --dir {DIR}/escape

> En física, los **criterios** de verificación los escribe el modelo antes de resolver (y se muestran en el registro). **Revísalos**: la verificación es tan fiable como esos criterios.

## 5. Ver el progreso y el resultado

In [ ]:
!python demostrador.py estado --dir {DIR}/par_mayor
print()
!python demostrador.py estado --dir {DIR}/escape

In [ ]:
print(open(f'{DIR}/par_mayor/resultado.lean').read())

## 6. Problemas largos o difíciles

- Dale **horas**: `--horas 3` (límite de la sesión; al llegar, se pausa con el progreso guardado).
- Más intentos y más profundidad de descomposición: `--directos 10 --replanes 5 --profundidad 3`.
- Déjalo en segundo plano y consulta el registro (se puede cerrar el navegador si usas Drive y reanudas después):

In [ ]:
!nohup python demostrador.py resolver --problema problemas/facil_par_mayor.json --modelo {MODELO} --lean {LEAN} --dir {DIR}/par_mayor --horas 3 > {DIR}/par_mayor.log 2>&1 &

In [ ]:
!tail -n 20 {DIR}/par_mayor.log

Para problemas **con Mathlib** (números reales, análisis...) hace falta un proyecto *lake* con Mathlib (descarga grande, ~5 GB de caché); después: `--lake-dir RUTA --mathlib`.

Los problemas más ambiciosos (agujeros negros, espacio-tiempo) se añaden como archivos `.json` en `problemas/` (`enunciado`, `backend`, `dominio`).